# SignalProcessing-01 — Le signal dans le domaine fréquentiel

**Navigation** : [SignalProcessing-02 — La convolution et le filtrage →](SignalProcessing-02-Convolution-Et-Filtrage-Python.ipynb) | [Index de la série](../README.md)

**Série SignalProcessing** · Carnet 01 — Le signal dans le domaine fréquentiel

Un signal — une mesure, un son, une onde — est d'abord une liste de valeurs dans
le **temps**. Mais une grande partie de ce qu'on veut en faire (le filtrer, le
compresser, en repérer les composantes) se fait plus naturellement dans le
**domaine des fréquences**. Ce carnet exécute, avec l'organe natif `np.fft`,
le passage d'un domaine à l'autre, et montre *ce que chaque brique fait* sur des
signaux réels : où se cache l'énergie d'un signal, comment un échantillonnage
trop lent invente une fausse fréquence, comment on supprime une composante, et
pourquoi convoluer dans le temps, c'est multiplier dans le spectre.


## Enjeu — du temps à la fréquence

Une sinusoïde « à 50 Hz » n'a rien d'abstrait : c'est l'information que, dans
le signal, toute l'énergie se concentre à une fréquence précise. Le traitement
du signal numérique transforme cette intuition en **calcul** : on passe le signal
du domaine temps au domaine fréquence (la transformée de Fourier), on y opère,
puis on revient. Tout repose sur un seul objet — la **FFT** — et sur les
propriétés qu'elle a.


## 0. La transformée de Fourier (FFT)

La **transformée de Fourier discrète** d'un signal `x` (échantillonné à `fs` Hz)
est le tableau `fft(x)` : chaque case donne, pour une fréquence donnée, combien
d'énergie le signal y porte. Pour un signal **réel**, le spectre est symétrique
(on lit la moitié positive).


In [1]:
import numpy as np

# --- La transformee de Fourier discrete (FFT) ---
# Un signal : une sinusoide pure a 50 Hz, duree 1 s, echantillonnage 1000 Hz.
fs = 1000.0                          # frequence d'echantillonnage (Hz)
N  = 1000                            # nombre d'echantillons
t  = np.arange(N) / fs
signal = np.sin(2 * np.pi * 50.0 * t)

# FFT REEL (rfft) : passe du domaine TEMPS au domaine FREQUENCE, et ne garde
# que la moitie positive du spectre (un signal reel est symetrique).
S     = np.fft.rfft(signal)
freqs = np.fft.rfftfreq(N, d=1.0 / fs)   # axe des frequences, 0 .. fs/2 (Hz)
mag   = np.abs(S)

# Le pic du spectre est exactement a 50 Hz :
f_max = freqs[np.argmax(mag)]
print(f"Frequence du pic du spectre : {f_max:.1f} Hz")
print(f"Points du spectre positif   : {len(S)} (la moitie positive suffit, signal reel)")
assert abs(f_max - 50.0) < 1e-9, "une sinusoide a 50 Hz donne un pic a 50 Hz"
print("=> la FFT dit OU se cache l'energie du signal, en hertz.")


Frequence du pic du spectre : 50.0 Hz
Points du spectre positif   : 501 (la moitie positive suffit, signal reel)
=> la FFT dit OU se cache l'energie du signal, en hertz.


### Lecture du résultat

Le spectre d'une sinusoïde à 50 Hz a **un seul pic, et il est à 50 Hz** — pas
à 49, pas à 51. La FFT ne « ressemble à » la fréquence : elle la **localise
exactement**. C'est la brique de base : à partir d'elle on filtre, on mesure,
on comprime.


## 1. Le théorème de Nyquist et l'alias

Pour **représenter** fidèlement un ton de fréquence `f`, il faut l'échantillonner
à **plus de `2·f`** (Nyquist). En dessous, le ton ne disparaît pas : il se
**ré-invente** à une fausse fréquence plus basse — l'**alias**. C'est pourquoi
une vidéo mal échantillonnée fait « rétro-pédaler » les roues d'une voiture.


In [2]:
# --- Le theoreme de Nyquist et l'alias ---
# Nyquist : pour representer un ton a f, il faut echillonner a plus de 2*f.
# Sinon le ton se "re-invente" a une fausse frequence (l'alias).

def frequence_vue(freq_ton, fs):
    """Frequence a laquelle on VOIT un ton de freq_ton si on echillonne a fs."""
    alias = abs(freq_ton % fs)          # repli par la periode d'echantillonnage
    return min(alias, fs - alias)       # repli dans [0, fs/2] (Nyquist)

# Un ton a 400 Hz :
#   bien echillonne  (fs = 1000 Hz, Nyquist = 500 > 400) : vu a 400 Hz.
#   mal echillonne   (fs = 700  Hz, Nyquist = 350 < 400) : alias a 300 Hz.
print(f"Ton 400 Hz, fs = 1000 Hz (OK)       -> vu a {frequence_vue(400, 1000):.0f} Hz")
print(f"Ton 400 Hz, fs = 700  Hz (trop lent) -> vu a {frequence_vue(400, 700):.0f} Hz")

# Verifie par la FFT : on genere le ton a 400 Hz, on l'echillonne a 700 Hz,
# et on regarde a quelle frequence tombe le pic.
fs_bad = 700.0
N_bad  = 700
t_bad  = np.arange(N_bad) / fs_bad
x_bad  = np.sin(2 * np.pi * 400.0 * t_bad)
S_bad  = np.abs(np.fft.rfft(x_bad))
f_vu   = np.fft.rfftfreq(N_bad, d=1.0 / fs_bad)[np.argmax(S_bad)]
print(f"Mesure par FFT : le pic tombe a {f_vu:.0f} Hz")
assert abs(f_vu - 300.0) < 1e-9, "sous-echantillonnage : 400 Hz devient un alias a 300 Hz"
print("=> echillonner trop lentement invente une fausse frequence plus basse.")


Ton 400 Hz, fs = 1000 Hz (OK)       -> vu a 400 Hz
Ton 400 Hz, fs = 700  Hz (trop lent) -> vu a 300 Hz
Mesure par FFT : le pic tombe a 300 Hz
=> echillonner trop lentement invente une fausse frequence plus basse.


### Lecture du résultat

Le ton de **400 Hz** échantillonné à 700 Hz (Nyquist 350 Hz, insuffisant)
n'apparaît plus à 400 Hz : le pic du spectre tombe à **300 Hz** (`700 − 400`).
L'information n'est pas perdue, elle est **trompée** — on ne peut plus
distinguer un vrai 300 Hz d'un 400 Hz mal échantillonné. D'où la règle
opérationnelle : échantillonner **au moins deux fois** la plus haute fréquence
présente.


## 2. Filtrer, c'est choisir dans le spectre

Un **filtre passe-bas** laisse passer les basses fréquences et coupe les hautes.
Dans le domaine fréquentiel, c'est trivial : on **zeroe** les cases du spectre
au-delà du seuil, puis on revient dans le temps. On le démontre sur un signal à
deux composantes (50 Hz + 200 Hz) : on supprime la 200 Hz, on garde la 50 Hz.


In [3]:
# --- Filtrage passe-bas dans le domaine frequentiel ---
# Un signal = 50 Hz + 200 Hz. On "zeroe" les frequences au-dela de 100 Hz
# dans le spectre, puis on revient dans le temps : le 200 Hz disparaît.

fs    = 1000.0
t     = np.arange(0.0, 1.0, 1.0 / fs)
signal = np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 200 * t)   # deux tons

S     = np.fft.fft(signal)
freqs = np.fft.fftfreq(len(signal), d=1.0 / fs)

# Masque passe-bas : garder |f| <= 100 Hz, zeroer le reste.
masque   = np.abs(freqs) <= 100.0
S_filtre = np.where(masque, S, 0.0)
filtre   = np.real(np.fft.ifft(S_filtre))

def energie_a(freq, sig):
    Sp = np.abs(np.fft.fft(sig))
    return Sp[np.argmin(np.abs(freqs - freq))]   # bin le plus proche de freq

e50_avant, e200_avant = energie_a(50.0, signal), energie_a(200.0, signal)
e50_apres,  e200_apres = energie_a(50.0, filtre), energie_a(200.0, filtre)
print(f"Energie a 50  Hz : avant {e50_avant:8.1f}  ->  apres {e50_apres:8.1f}  (gardee)")
print(f"Energie a 200 Hz : avant {e200_avant:8.1f}  ->  apres {e200_apres:8.3f}  (supprimee)")
assert e200_apres < 1e-6 * e200_avant, "le filtre supprime la composante 200 Hz"
assert e50_apres  > 0.5 * e50_avant,   "le filtre garde la composante 50 Hz"
print("=> filtrer = choisir, dans le spectre, ce qu'on garde.")


Energie a 50  Hz : avant    500.0  ->  apres    500.0  (gardee)
Energie a 200 Hz : avant    500.0  ->  apres    0.000  (supprimee)
=> filtrer = choisir, dans le spectre, ce qu'on garde.


### Lecture du résultat

La composante **200 Hz** est annulée (son énergie tombe à ~0), la **50 Hz**
survit intacte. Le filtre ne « détruit » rien dans le temps : il **sélectionne**
des fréquences dans le spectre. C'est le geste qui, multiplié, fait un égaliseur,
un débruitage, une compression.


## 3. La fuite spectrale et la fenêtre de Hann

Un signal qu'on analyse est toujours **tronqué** à une fenêtre finie. Si le ton
ne complète pas un **nombre entier de cycles** dans cette fenêtre, son énergie
**fuit** sur les fréquences voisines (fuite spectrale) : le pic s'élargit.
Multiplier le signal par une **fenêtre de Hann** (qui ramène doucement les bords
à zéro) concentre l'énergie de nouveau sur le pic.


In [4]:
# --- La fuite spectrale et la fenetre de Hann ---
# Une sinusoide qui ne complete pas un entier de cycles dans la fenetre "fuit"
# son energie sur les frequences voisines. Une fenetre de Hann la concentre.

fs    = 1000.0
N     = 1000                                   # fenetre de 1 s
t     = np.arange(N) / fs
f_ton = 50.5                                   # 50.5 cycles : NON entier -> fuite
x     = np.sin(2 * np.pi * f_ton * t)

def ratio_pic_cotes(sig):
    """Pic principal / energie des 3 bins de part et d'autre (les 'cotes')."""
    S   = np.abs(np.fft.rfft(sig))
    i   = np.argmax(S)
    cotes = S[max(0, i-3):i].sum() + S[i+1:i+4].sum()
    return S[i] / (cotes + 1e-12)

r_rect = ratio_pic_cotes(x)                    # fenetre rectangulaire (implicite)
r_hann = ratio_pic_cotes(x * np.hanning(N))    # fenetre de Hann
print(f"Ratio pic / cotes, fenetre RECTANGULAIRE : {r_rect:.2f}")
print(f"Ratio pic / cotes, fenetre de HANN       : {r_hann:.2f}")
assert r_hann > r_rect, "la fenetre de Hann concentre l'energie (moins de fuite)"
print("=> Hann ecrase les bords : le pic redevient net, la fuite chute.")


Ratio pic / cotes, fenetre RECTANGULAIRE : 0.45
Ratio pic / cotes, fenetre de HANN       : 0.68
=> Hann ecrase les bords : le pic redevient net, la fuite chute.


### Lecture du résultat

Avec une fenêtre **rectangulaire**, le ton à 50,5 Hz (non entier de cycles)
étale son énergie sur les bins voisins — le ratio pic/côtes est bas. La fenêtre
de **Hann** ramène les bords à zéro sans coupure : l'énergie se reconcentre sur
le pic, le ratio monte. C'est le compromis classique — Hann **résout mieux** les
fréquences proches, au prix d'une amplitude atténuée.


## 4. Le théorème de convolution

Le résultat le plus utile de tout : **convoluer** deux signaux dans le **temps**
équivaut à **multiplier** leurs spectres dans la **fréquence**. C'est ce qui rend
la FFT si puissante — une convolution coûte `O(n²)` à la main, mais `O(n log n)`
via `ifft(fft(a) · fft(b))`.


In [5]:
# --- Le theoreme de convolution ---
# Convoluer deux signaux dans le TEMPS = MULTIPLIER leurs spectres dans la
# FREQUENCE. Une convolution "lente" devient une multiplication "rapide".

a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([0.5, 1.0, 0.5, 0.0])        # meme longueur (padding a zero)

def convolution_circulaire(u, v):
    n, out = len(u), np.zeros(len(u))
    for i in range(n):
        for j in range(n):
            out[i] += u[j] * v[(i - j) % n]
    return out

via_main = convolution_circulaire(a, b)                 # a la main, dans le temps
via_fft  = np.real(np.fft.ifft(np.fft.fft(a) * np.fft.fft(b)))   # via la FFT

print("Convolution (methode directe) :", np.round(via_main, 4))
print("Convolution (via FFT)         :", np.round(via_fft, 4))
np.testing.assert_allclose(via_fft, via_main, atol=1e-9)
print("=> les deux concordent : le theoreme de convolution est verifie.")


Convolution (methode directe) : [6. 4. 4. 6.]
Convolution (via FFT)         : [6. 4. 4. 6.]
=> les deux concordent : le theoreme de convolution est verifie.


### Lecture du résultat

Les deux voies — convolution **à la main** dans le temps, et **multiplication**
des spectres suivie d'une FFT inverse — donnent **le même résultat** à la
précision numérique près. C'est le fondement des filtres rapides, de la
corrélation, et de la plupart des traitements « en fréquence ».


## Synthèse — le signal, vu de la fréquence

| Brique | Ce qu'on a montré |
|---|---|
| FFT | une sinusoïde à 50 Hz → un pic **unique à 50 Hz** dans le spectre |
| Nyquist | un ton 400 Hz échantillonné à 700 Hz → **alias à 300 Hz** |
| Filtre passe-bas | zeroer le spectre au-delà du seuil **supprime** la composante haute |
| Fenêtre de Hann | concentre l'énergie, **réduit la fuite spectrale** |
| Convolution | convoluer (temps) = **multiplier les spectres** (fréquence) |

Le fil rouge : **passer dans le domaine fréquence, opérer y, revenir.** Presque
tout le traitement du signal — filtrage, compression, détection — est une
variantes de ce geste.


## Exercices


In [6]:
# Exercice 1 — La phase
# La FFT renvoie des nombres COMPLEXES : une magnitude (l'amplitude) ET une
# phase. Ecrire pourquoi la phase est indispensable pour reconstruire le signal
# (ifft), et a quoi elle sert en pratique (synchronisation, correlation, ...).
def exercice_phase():
    # TODO etudiant
    return None  # TODO etudiant


In [7]:
# Exercice 2 — Le filtre passe-haut / passe-bande
# Le passe-bas zeroe les HAUTES frequences. Parite : ecrire le PASSE-HAUT
# (garder |f| >= seuil) et le PASSE-BANDE (garder une plage [a, b]), en
# reutilisant le masque du carnet.
def exercice_passe_haut():
    # TODO etudiant
    return None  # TODO etudiant


In [8]:
# Exercice 3 — Le SNR (rapport signal / bruit)
# Mesurer le SNR : le rapport (en dB) entre l'energie du signal utile et celle
# du bruit, avant / apres un filtrage. Ecrire la formule
# 10*log10(P_sig / P_bruit) et pourquoi le filtrage le fait monter.
def exercice_snr():
    # TODO etudiant
    return None  # TODO etudiant


## À retenir

- **La FFT** localise **exactement** les fréquences d'un signal (une sinusoïde
  à 50 Hz → un pic à 50 Hz).
- **Nyquist** : échantillonner **au moins deux fois** la plus haute fréquence,
  sinon le ton **alias** à une fausse fréquence plus basse.
- **Filtrer** = **zeroer** les fréquences indésirables dans le spectre, puis
  revenir dans le temps.
- **La fenêtre de Hann** concentre l'énergie sur le pic et **réduit la fuite
  spectrale** d'un signal tronqué.
- **Convoluer** dans le temps = **multiplier** dans le spectre — le geste qui
  rend tout « en fréquence » rapide.


## Bibliographie

- A. V. Oppenheim, R. W. Schafer, *Discrete-Time Signal Processing*, 3ᵉ éd.,
  Prentice Hall, 2010 — le référentiel du traitement du signal numérique.
- M. Vetterli, K. Kovačević, V. Goyal, *Fourier and Wavelet Signal
  Processing*, Cambridge University Press, 2013 —
  le passage du temps à la fréquence.
- [Carnet 02 — Convolution et filtrage](SignalProcessing-02-Convolution-Et-Filtrage-Python.ipynb) : les filtres réels — FIR/IIR, la réponse en fréquence.
